# Lab 8: Report

## Charpy Impact Testing

The main objective of this lab is to understand and explain fracture under severe loading conditions (high loading rate or extreme temperatures). In this notebook you will:

- Plot the impact energy of four metals as a function of testing temperature.
- Determine the ductile-to-brittle transition temperature (DBTT) of the two steels.
- Describe how the fracture mode of each material changes with temperature.

Recommended reading: Callister, *Materials Science and Engineering: An Introduction*, Section 8.6.

## 1. Enter the data into the notebook

In Lab 7 you imported a large `.csv` file with code. This lab has only a few measurements, so this time you will **type the data into a pandas DataFrame by hand**.

1. Open `ImpactEnergy.csv` (in Excel or a text editor) and inspect it.
    - Each row is one testing temperature.
    - Each material has its own column of impact energies.
    - Not every material was tested at every temperature. The steels were tested at 0 °C and the brass and aluminum at room temperature (RT, 20 °C).
1. Note the **units** of the temperatures and the impact energies, and the **calibration value** at the bottom of the file.

Run the cell below to import the necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit,root_scalar

### 1.1 Building a DataFrame by hand

A DataFrame can be built from a **dictionary**, written inside curly brackets `{ }`:

- each **key** (the text before the colon) becomes a column name;
- each **value** (after the colon) is a list, inside square brackets `[ ]`, holding that column's entries from top to bottom;
- every list must have the **same length**, because each position in the lists is one row of the table;
- use `np.nan` ("not a number") for an empty cell, i.e. a measurement that was not made.

Run the example below and compare the code with the table it produces.

In [ ]:
# Example: a small table of made-up measurements
example = pd.DataFrame({
    "Time (s)": [0, 10, 20, 30],
    "Length (mm)": [50.0, 50.4, np.nan, 51.1],     # no measurement was made at 20 s
})

display(example)

### 1.2 Enter your impact data

Complete the code below with the values from `ImpactEnergy.csv`.

- Make one row for **every** testing temperature in the file, in the same order for every column.
- Enter the impact energies **exactly as recorded**, in ft-lb. You will convert the units in Section 2.
- Use `np.nan` wherever a material was not tested at that temperature.
- Enter the calibration value from the bottom of the file in `CALIBRATION`.

In [ ]:
# STUDENT: enter the data from ImpactEnergy.csv. Temperatures in °C, impact energies in ft-lb.
df = pd.DataFrame({
    "T / °C":        [],
    "1018 steel":    [],
    "1045 steel":    [],
    "2024 aluminum": [],
    "70/30 brass":   [],
})
df.set_index("T / °C",inplace=True)

# STUDENT: enter the calibration (offset) value in ft-lb measured without sample
CALIBRATION = 

# STUDENT: enter the conversion factor to convert from ft-lb to Joules
J_per_ftlb = 

# shift all data except temperature by the calibration value and convert to Joules
df -= CALIBRATION
df *= J_per_ftlb

display(df)

## 2. Plot impact energy versus testing temperature (20 points)

First, set your plotting choices. Give each of the four materials a unique color, marker, line style, or a combination. The names must match the column names in `MATERIALS`.

In [ ]:
# STUDENT: define a color, marker, and line style for each material.
COLOR     = dict(zip(df.columns,["", "", "", ""]))
MARKER    = dict(zip(df.columns,["", "", "", ]))
LINESTYLE = dict(zip(df.columns,["", "", "", ]))

### Plot the impact energy of all materials as a function of the testing temperature

Plot the data points with no connecting lines between them.

**Variables available inside the loop** (for one material at a time):

- `material`: name of the current material
- `series.index`: x values (temperatures)
- `series.values`: y values (impact energies)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for material,series in df.items():
    # STUDENT: plot impact energy vs temperature. Use the color, marker and line style defined previously

    # remove Nan values before plotting
    series = series.dropna()
    
    ax.plot(
        
    )

# STUDENT: include the axes labels with proper units, title, grid, and legend


plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption. Use the caption to explain the observed trends.
print("Figure 1. YOUR RESPONSE")

## 3. Ductile-to-brittle transition temperature of the steels (15 points)

The handout notes that the impact-energy curves of the steels should be S-shaped and are usually fitted with a **sigmoidal function**.

A common choice for Charpy data is the hyperbolic tangent. This fit is recommended by NIST:

$$
E(T) = \frac{E_\text{upper}+E_\text{lower}}{2} +\frac{E_\text{upper}-E_\text{lower}}{2} \, \tanh\left(\frac{T-\text{DBTT}}{A}\right)
$$

- $E(T)$: impact energy at testing temperature $T$
- $E_\text{upper}$: the **upper shelf** energy (ductile behavior at high temperature)
- $E_\text{lower}$: the **lower shelf** energy (brittle behavior at low temperature)
- $\text{DBTT}$: the temperature at the center of the transition
- $A$: temperature range corresponding to the half-width of the transition region 



In [ ]:
# STUDENT: encode the sigmoid function E(T) defined above using np.tanh
def sigmoid(T, E_lower, E_upper, DBTT, A):
    return       # for ex.: E_lower*E_upper-DBTT/A


### 3.1 Determine the steel ductile-to-brittle transition temperatures

The `curve_fit` method from the `scipy.optimize` module allows the user to fit `x_data` and `y_data` with a user defined function.
Define `sigmoid()` below to determine the fitting parameters.

Use Figure 1 to make reasonable estimates of $E_\text{upper}$, $E_\text{lower}$, $\text{DBTT}$, and $A$ and list them in the `INITIAL_GUESS`variable.

For datasets with fewer than 8 datapoints, NIST recommends fitting only the values of $\text{DBTT}$ and $A$.
Hence, the `lower shelf` and `upper shelf` values are not fitted but will stay as in your input.

In [ ]:
fit_parameters = {}

# STUDENT: enter intial guesses for the fitting parameters
INITIAL_GUESS = {
    "1018 steel": {"lower shelf": 0,
                   "upper shelf": 0,
                   "DBTT": 0,
                   "A": 0 },
    "1045 steel": {"lower shelf": 0,
                   "upper shelf": 0,
                   "DBTT": 0,
                   "A": 0},
                 }

# determine the sigmoidal fit parameters for each steel
for steel,guess in INITIAL_GUESS.items():
    series = df[steel].dropna()      # isolate the series of each steel
    
    # determine all four adjustable parameters with the curve_fit function
    popt, _ = curve_fit( sigmoid,
                         series.index, series.values,
                         p0=list(guess.values()),
                         maxfev=10000 )
    # E_atDBTT = fixed_shelves_sigmoid(popt[0], *popt)
    
    fit_parameters[steel] = popt

# store the values in a pandas dataframe
sigmoid_fit = pd.DataFrame(
    fit_parameters,
    index=["E_lower / J", "E_upper / J", "DBTT / °C", "A / °C",]
)

display(sigmoid_fit.round(2))

Next, use the fitted curve to estimate the temperature at which each steel has a fracture energy of 20 Joules.

In [ ]:
# Find the temperature where each fitted steel curve reaches 20 J.
target_energy = 20
results = []

# Use `root_scalar` to determine the temperature at which each steel has a fracture energy of 20 Joules
for steel,parms in sigmoid_fit.items():
    results.append(root_scalar(lambda T,args: sigmoid(T,*args)-target_energy,
                                bracket=[-200, 200],
                                args=parms,
                                method='brentq').root)

# append the 20 Joule fracture energy temperatures to the dataframe
sigmoid_fit.loc[f"DBTT (E = {target_energy} J) / °C"] = results
display(sigmoid_fit.round(2))

### 3.2 Plot the Sigmoidal fits and estimated DBTTs

Run the code below to plot the steel data together with the fitted curves and the two DBTT estimates. Complete the title, axis labels and caption.


In [ ]:
T_sampling = np.linspace(-200,200,200)

fig, ax = plt.subplots(figsize=(9, 6))

for steel,params in sigmoid_fit.items():
    series = df[steel]
    
    # STUDENT: plot your measured data
    ax.scatter(series.index,
               series.values,
               color=COLOR[steel],
               marker=MARKER[steel],
               label=f"{steel} data",
              )

    # fitted sigmoid curve
    ax.plot(T_sampling,
            sigmoid(T_sampling, *params[:-1]),
            color=COLOR[steel],
            linestyle=LINESTYLE[steel],
            label=f"{steel} sigmoid fit",
           )

    # DBTT estimates: at 20 J (filled star) and at the midpoint energy (open star)
    ax.scatter(params[f'DBTT (E = {target_energy} J) / °C'],
               sigmoid(params[f'DBTT (E = {target_energy} J) / °C'],*params[:-1]),
               marker="*",
               s=150,
               color=COLOR[steel],
               label=f"{steel} DBTT at {target_energy} J",
              )

    ax.scatter(params[f'DBTT / °C'],
               sigmoid(params[f'DBTT / °C'],*params[:-1]),
               marker="*",
               s=150,
               edgecolor='black',
               color=COLOR[steel],
               label=f"{steel} DBTT at midpoint",
            )   

# STUDENT: include the axes labels, title, grid, and legend



# This adds a secondary axis to also show the original ft-lb scale 
ax2 = ax.secondary_yaxis('right', functions=(lambda x:x/J_per_ftlb, lambda y:y*J_per_ftlb))
ax2.set_ylabel("Impact energy / ft-lb", color="gray")
ax2.tick_params(axis='y', labelcolor="gray")

plt.legend()
plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 2. YOUR RESPONSE")

**`YOUR RESPONSE`**: list the temperature range over which each steel transitions from ductile to brittle behavior and temperature at which each steel has a fracture energy of 20 Joules.

| Steel | Transition temperature range / °C | DBTT at 20 J / °C | DBTT at Midpoint |
|:---|:---:|:---:|:---:|
| 1018 steel | YOUR RESPONSE | YOUR RESPONSE | YOUR RESPONSE |
| 1045 steel | YOUR RESPONSE | YOUR RESPONSE | YOUR RESPONSE |



## 4. DBTT from the average energy (10 points)

**`YOUR RESPONSE`**: Does the DBTT change significantly when you choose the average between the high and the low impact energy levels instead of 20 J as the value to determine the DBTT? Why or why not?

*Replace this text with your response in paragraph form.*

## 5. Fracture surface micrographs and fracture mode (20 points)

**`YOUR RESPONSE`**: Describe and comment on the changes in fracture mode of the specimens for all materials.

Refer to macroscopic deformation observations, macroscopic fracture surface images (as available), and SEM micrographs available on D2L. Call out specific images by their labels, and relate the fracture mode you see to the impact energies in Figure 1.

*Replace this text with your response in paragraph form.*


In [ ]:
# Use this code cell to display the relevant images to support your discussion
# Refer to the lab7 report to see how to include images in your jupyter notebook
# Remember to include captions for each figure

## CONTRIBUTION STATEMENT

*List each contributor and what they did.*

**`YOUR RESPONSE`**

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. If the PDF compile fails check your $\LaTeX$ code for errors.